# Unitary Manifold — Notebook 03: Multiverse Fixed-Point (FTUM)

This notebook demonstrates **Pillar 5 — Multiverse Topology** and the
**Final Theorem of the Unitary Manifold (FTUM)**:

> There exists a fixed point Ψ* of the combined operator
> U = I + H + T (Irreversibility + Holography + Topology)
> such that UΨ* = Ψ*.

We iterate `U` on chain and fully-connected multiverse networks and show
convergence, residual decay, and entropy equilibration.

**Repository:** https://github.com/wuzbak/Unitary-Manifold-

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

import numpy as np
import matplotlib.pyplot as plt

from src.multiverse.fixed_point import (
    MultiverseNetwork, MultiverseNode,
    fixed_point_iteration,
    apply_irreversibility, apply_holography, apply_topology
)

plt.rcParams['figure.dpi'] = 100

## 1 · Chain network — FTUM convergence

In [ ]:
rng = np.random.default_rng(42)
chain = MultiverseNetwork.chain(n=5, coupling=0.05, rng=rng)

print("Chain network — initial node entropies:")
for i, node in enumerate(chain.nodes):
    print(f"  Node {i}: S={node.S:.4f}, A={node.A:.4f}, Q_top={node.Q_top:.4f}")

result_chain, residuals_chain, converged_chain = fixed_point_iteration(
    chain, max_iter=500, tol=1e-6
)

print(f"\nConverged: {converged_chain}  after {len(residuals_chain)} iterations")
print(f"Final residual: {residuals_chain[-1]:.2e}")

## 2 · Fully-connected network — FTUM convergence

In [ ]:
rng2 = np.random.default_rng(42)
full = MultiverseNetwork.fully_connected(n=5, coupling=0.05, rng=rng2)

result_full, residuals_full, converged_full = fixed_point_iteration(
    full, max_iter=500, tol=1e-6
)

print(f"Converged: {converged_full}  after {len(residuals_full)} iterations")
print(f"Final residual: {residuals_full[-1]:.2e}")

## 3 · Residual decay — chain vs fully-connected

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(residuals_chain, label='Chain (n=5, c=0.05)', color='royalblue')
ax.semilogy(residuals_full,  label='Fully connected (n=5, c=0.05)', color='darkorange', linestyle='--')
ax.axhline(1e-6, color='grey', linestyle=':', linewidth=0.8, label='tol = 1e-6')
ax.set_xlabel('Iteration')
ax.set_ylabel('‖Ψⁿ⁺¹ − Ψⁿ‖')
ax.set_title('FTUM Fixed-Point Convergence  (U = I + H + T)')
ax.legend()
plt.tight_layout()
plt.show()

## 4 · Entropy equilibration across nodes

In [ ]:
# Track per-node entropy through iteration
from src.multiverse.fixed_point import _apply_U

rng3 = np.random.default_rng(42)
net = MultiverseNetwork.chain(n=5, coupling=0.05, rng=rng3)

n_track = 300
S_history = np.zeros((n_track + 1, net.n_nodes()))

for i, node in enumerate(net.nodes):
    S_history[0, i] = node.S

for step_i in range(n_track):
    net = _apply_U(net, dt=1e-3)
    for i, node in enumerate(net.nodes):
        S_history[step_i + 1, i] = node.S

fig, ax = plt.subplots(figsize=(9, 5))
for i in range(net.n_nodes()):
    ax.plot(S_history[:, i], label=f'Node {i}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Entropy S')
ax.set_title('Entropy equilibration across chain multiverse nodes')
ax.legend()
plt.tight_layout()
plt.show()

print("Final node entropies:", [f"{node.S:.4f}" for node in net.nodes])

## 5 · Coupling strength scan

In [ ]:
couplings = [0.01, 0.05, 0.10, 0.20, 0.50]
iter_to_converge = []

for c in couplings:
    rng_c = np.random.default_rng(42)
    net_c = MultiverseNetwork.chain(n=5, coupling=c, rng=rng_c)
    _, res, conv = fixed_point_iteration(net_c, max_iter=500, tol=1e-6)
    iter_to_converge.append(len(res) if conv else 500)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(couplings, iter_to_converge, 'o-', color='seagreen')
ax.set_xlabel('Coupling strength')
ax.set_ylabel('Iterations to convergence')
ax.set_title('Convergence speed vs. inter-manifold coupling')
plt.tight_layout()
plt.show()